Лабораторна робота: Аналіз маршруту на основі даних OpenStreetMap
Мета:
Навчитися використовувати дані з OpenStreetMap для побудови картографічних відображень, а також для аналізу маршрутів між двома точками.
Обладнання та програмне забезпечення:
•	Комп'ютер з доступом до інтернету
•	Встановлений Python
•	Встановлені бібліотеки: osmnx, networkx, matplotlib, geopandas, shapely

Кроки виконання:
1.	Завантажити карту довільного міста з OpenStreetMap.

In [1]:
import osmnx as ox
import folium
import json
import copy
import networkx as nx
from shapely.geometry import LineString
from networkx.algorithms.simple_paths import shortest_simple_paths

ModuleNotFoundError: No module named 'osmnx'

In [ ]:
#load city with only drive roads
G = ox.graph_from_place('Odesa, Ukraine', network_type='drive')

#show it
ox.plot_graph(G)

In [ ]:
# Get the centroid coordinates of the nodes
nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
center_latlng = nodes.union_all().centroid.coords[0]
center_lat, center_lng = center_latlng[1], center_latlng[0]

#create folium interactive map
m = folium.Map(location=[center_lat, center_lng], zoom_start=12)

#Show it
m

2.	Відобразити вулиці та будинки на карті.

In [ ]:
#Load buildings data
tags = {"building": True}
buildings = ox.features_from_place("Odessa, Ukraine", tags)
buildings = buildings.to_crs(epsg=3857)

#Calculate area for each building geometry
buildings['area'] = buildings.geometry.area

n = 2500

#Select the top largest buildings by area
buildings = buildings.sort_values(by='area', ascending=False).head(n)
buildings = buildings.to_crs(epsg=4326)


#Create a Folium map centered on the graph
m = folium.Map(location=[center_lat, center_lng], zoom_start=13)

#Add buildings to the map
geojson_buildings = json.loads(buildings.to_json())

folium.GeoJson(
    geojson_buildings,
    name = f"Top {n} Largest Buildings",
    style_function=lambda feature: {
        'fillColor': 'orange',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.5,
    }
).add_to(m)

#Add road network to the map
edges = ox.graph_to_gdfs(G, nodes=False)
folium.GeoJson(edges).add_to(m)

#Add layer control and show the map
folium.LayerControl().add_to(m)

#Show the map
m



3.	Вибрати довільні два будинки.

In [ ]:
building1, building2 = buildings.sample(2, random_state=32).itertuples(index=False)
print(building1.geometry)
print(building2.geometry)

In [ ]:
#Centers of the buildings
centroid1 = building1.geometry.centroid
centroid2 = building2.geometry.centroid

#Find the nearest nodes
orig_node = ox.distance.nearest_nodes(G, centroid1.x, centroid1.y)
dest_node = ox.distance.nearest_nodes(G, centroid2.x, centroid2.y)

#Shortest routes
route = nx.shortest_path(G, orig_node, dest_node, weight='length')


4.	Побудувати та зобразити на карті найкоротшу відстань між цими двома будинками.
5.	Відобразити цей маршрут на фоновій географічній карті.

In [ ]:
m = folium.Map(location=[center_lat, center_lng], zoom_start=13)

In [ ]:
#Plot the Graph
fig, ax = ox.plot_graph_route(
    G,                 
    route,            
    route_color='red',
    route_linewidth=4,
    node_size=0,
    bgcolor='white'
)

In [ ]:
#route geo creation 
route_nodes = [G.nodes[node] for node in route]
route_coords = [(node['x'], node['y']) for node in route_nodes]
route_line = LineString(route_coords)

#Add the route as a GeoJSON layer to the map
folium.GeoJson(
    data={"type": "Feature", "geometry": route_line.__geo_interface__},
    name="Shortest Route",
    style_function=lambda feature: {
        'color': 'red',
        'weight': 5,
        'opacity': 0.8,
    }
).add_to(m)

#Show it on the map
m

6.	Розрахувати та зобразити альтернативні маршрути

In [ ]:
#number of routes to generate
k = 7

#Function to generate k diverse routes using edge penalty
def k_diverse_routes(G, orig_node, dest_node, k=k, weight='length', penalty_factor=3.0):

    G_modified = copy.deepcopy(G)  # preserve original graph
    routes = []

    for i in range(k):
        try:
            route = nx.shortest_path(G_modified, orig_node, dest_node, weight=weight)
            routes.append(route)

            # Penalize the edges used in this route
            for u, v in zip(route[:-1], route[1:]):
                if G_modified.has_edge(u, v):
                    for key in G_modified[u][v]:
                        if weight in G_modified[u][v][key]:
                            G_modified[u][v][key][weight] *= penalty_factor
        except nx.NetworkXNoPath:
            print(f"Route #{i+1}: no path found.")
            break

    return routes

#Generate diverse routes
routes = k_diverse_routes(G, orig_node, dest_node, k=k, penalty_factor=3.0)

#Define the color for routes
color_cycle = ['black', 'red', 'blue']

#Initialize the folium map centered on the building
m = folium.Map(location=[centroid1.y, centroid1.x], zoom_start=15)

#Add each route to the map with rotating colors
for i, route in enumerate(routes):
    gdf = ox.routing.route_to_gdf(G, route, weight='length')

    coords = []
    for geom in gdf.geometry:
        coords.extend([(lat, lon) for lon, lat in geom.coords])

    folium.PolyLine(
        coords,
        color=color_cycle[i % len(color_cycle)],
        weight=4,
        opacity=0.7,
        tooltip=f'Route {i+1}',
        name=f'Route {i+1}'
    ).add_to(m)


#Add building markers
folium.Marker(
    [centroid1.y, centroid1.x],
    tooltip="Building 1",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    [centroid2.y, centroid2.x],
    tooltip="Building 2",
    icon=folium.Icon(color="blue")
).add_to(m)

#Add a layer control to toggle routes
folium.LayerControl().add_to(m)

#Show the final map
m


7.	Вибрати довільних 10 історико-культурних місць на карті міста, та спланувати оптимальний маршрут від вокзалу та з поверненням на вокзал (задача комівояжера). Вивести маршрут на карту міста.

In [ ]:
G = ox.graph_from_place("Odesa, Ukraine", network_type="walk", simplify=True)

buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])]

# show unique buildings
print(buildings['building'].unique())

In [ ]:
#select tags
tags = {
    'historic': True,
    'tourism': 'museum',
    'building': ['church', 'theatre'],
    'amenity': ['arts_centre', 'theatre', 'community_centre']
}

# Odesa 
gdf = ox.features_from_place("Odesa, Ukraine", tags=tags)
selected_places = gdf.sample(10, random_state=4)

In [ ]:
#Centers (lat, lon)
place_centroids = selected_places.to_crs(epsg=3857).geometry.centroid.to_crs(epsg=4326)
place_coords = [(geom.y, geom.x) for geom in place_centroids]

#Railway station
station_tags = {'building': 'train_station'}
station_gdf = ox.features_from_place("Odesa, Ukraine", station_tags)

#Choice first record
station = station_gdf.iloc[0]
station_centroid = station.geometry.centroid
station_coord = (station_centroid.y, station_centroid.x)


In [ ]:
# Fusion coordinates of places with railway station
all_coords = [station_coord] + place_coords

#Coords to nodes
nodes = [ox.distance.nearest_nodes(G, lon, lat) for lat, lon in all_coords]

disconnected_pairs = []
 
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        if not nx.has_path(G, nodes[i], nodes[j]):
            disconnected_pairs.append((nodes[i], nodes[j]))

if disconnected_pairs:
    print("Didn't find related nodes")
    for u, v in disconnected_pairs:
        print(f" - Node {u} doesn't have pass to{v}")
else:
    print("All nodes are related")


In [ ]:
# Center of city
center_lat, center_lon = station_coord

#Creat the map
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

#Add walk routes
edges = ox.graph_to_gdfs(G, nodes=False, edges=True)

for _, row in edges.iterrows():
    if isinstance(row.geometry, LineString):
        coords = [(lat, lon) for lon, lat in row.geometry.coords]
        folium.PolyLine(coords, color="blue", weight=1, opacity=0.5).add_to(m)

#Mark historical places
for i, (lat, lon) in enumerate(place_coords):
    name = selected_places.iloc[i].get("name", f"Obj {i+1}")
    folium.Marker(
        location=(lat, lon),
        popup=name,
        icon=folium.Icon(color="red", icon="university", prefix="fa")
    ).add_to(m)

#Mark railway station
folium.Marker(
    location=station_coord,
    popup="Railway statuion",
    icon=folium.Icon(color="green", icon="train", prefix="fa")
).add_to(m)

#Show the map
m


In [ ]:
#Greedy algorithm for finding the routes
def greedy_diverse_tsp(G, nodes_list, penalty_factor=3.0, weight="length"):
    G_mod = copy.deepcopy(G)
    remaining = nodes_list[1:]
    current = nodes_list[0]
    tsp_nodes = [current]

    while remaining:
        best_dist = float("inf")
        best_node = None
        best_path = None

        #Find the nearest node
        for target in remaining:
            try:
                path = nx.shortest_path(G_mod, current, target, weight=weight)
                dist = nx.shortest_path_length(G_mod, current, target, weight=weight)
                if dist < best_dist:
                    best_node = target
                    best_path = path
                    best_dist = dist
            except nx.NetworkXNoPath:
                continue

        #Add route
        tsp_nodes.extend(best_path[1:])
        remaining.remove(best_node)
        current = best_node

        # penalty to the used edges
        for u, v in zip(best_path[:-1], best_path[1:]):
            if G_mod.has_edge(u, v):
                for k in G_mod[u][v]:
                    if weight in G_mod[u][v][k]:
                        G_mod[u][v][k][weight] *= penalty_factor

    # Comeback to the railway station
    try:
        path_back = nx.shortest_path(G_mod, current, nodes_list[0], weight=weight)
        tsp_nodes.extend(path_back[1:])
    except nx.NetworkXNoPath:
        print("Route doesn't exist")

    return tsp_nodes

#Coords to nodes
all_coords = [station_coord] + place_coords
nodes = [ox.distance.nearest_nodes(G, lon, lat) for lat, lon in all_coords]

#Run algorithm
tsp_diverse_nodes = greedy_diverse_tsp(G, nodes)

#Showing
fig, ax = ox.plot_graph_route(G, tsp_diverse_nodes, route_linewidth=4, node_size=10, bgcolor="white")


In [ ]:
# Coords of nodes from G
node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in tsp_diverse_nodes}
route_coords = [node_coords[node] for node in tsp_diverse_nodes if node in node_coords]

#Creating map
center_lat, center_lon = station_coord
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

#Add route
folium.PolyLine(route_coords, color="blue", weight=4, opacity=0.8).add_to(m)

#Mark historical buildings
all_coords = [station_coord] + place_coords
node_to_coord = dict(zip(nodes, all_coords))

#Without repeats
seen = set()
coord_sequence = [coord for node in tsp_diverse_nodes
                  if (coord := node_to_coord.get(node)) and not (coord in seen or seen.add(coord))]

#Add markers
for idx, (lat, lon) in enumerate(coord_sequence):
    if idx == 0:
        folium.Marker(
            location=(lat, lon),
            popup="Railway station",
            icon=folium.Icon(color="green", icon="train", prefix="fa")
        ).add_to(m)
    else:
        name = selected_places.iloc[idx - 1].get("name", f"Object {idx}")
        folium.Marker(
            location=(lat, lon),
            popup=f"{idx}. {name}",
            icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(m)

#Show the map
m
